# AI 기본법 항 단위 Qdrant 테스트

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
from common.ai_model import get_embedding_model
from common.qdrant import get_qdrant_client
from qdrant_client.models import Distance, PointStruct, VectorParams
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
XML_PATH = PROJECT_ROOT / 'data' / 'ai_basic_law' / 'ai_basic_law.xml'
root = ET.parse(XML_PATH).getroot()

In [2]:
# 조문 단위 코드는 사용하지 않고 항 단위만 생성합니다.
chunks = []
for unit in root.findall('.//조문단위'):
    article_no = unit.findtext('조문번호', '').strip()
    title = unit.findtext('조문제목', '').strip()
    for paragraph_no, paragraph in enumerate(unit.findall('./항'), start=1):
        content = ' '.join(''.join(paragraph.itertext()).split())
        if content:
            chunks.append({
                'id': len(chunks) + 1,
                'chunk_type': 'paragraph',
                'article': f'제{article_no}조',
                'paragraph': f'{paragraph_no}항',
                'title': title,
                'content': content,
                'source': XML_PATH.name,
            })
print(f'항 단위 청크 수: {len(chunks)}개')

항 단위 청크 수: 166개


In [3]:
embedding_client = get_embedding_model()
qdrant_client = get_qdrant_client()
COLLECTION_NAME = 'ai_basic_law_paragraph_only_v1'
vectors = await embedding_client.aembed_documents([x['content'] for x in chunks])
if qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.delete_collection(COLLECTION_NAME)
qdrant_client.create_collection(collection_name=COLLECTION_NAME, vectors_config=VectorParams(size=len(vectors[0]), distance=Distance.COSINE))
points = [PointStruct(id=x['id'], vector=v, payload=x) for x, v in zip(chunks, vectors)]
qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f'{len(points)}개 항 청크 저장 완료')

CancelledError: 

In [ ]:
query = '고영향 인공지능이란 무엇인가요?'
query_vector = await embedding_client.aembed_query(query)
result = qdrant_client.query_points(collection_name=COLLECTION_NAME, query=query_vector, limit=5)
for point in result.points:
    print(f"[{point.payload['article']} {point.payload['paragraph']}] score={point.score:.4f}")
    print(point.payload['content'])
